### imports

In [ ]:
import os
import random
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, Input
from keras.utils import Sequence
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

### Parameters

In [ ]:
DATA_DIR = r'/home/arc/Code/eq_idn/oneforall'
CATEGORIES = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
              '+', '-', '*', '%', '[', ']']
IMG_SIZE = 28
BATCH_SIZE = 64
EPOCHS = 30
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

#### load images function

In [ ]:
def load_raw_data(data_dir, categories):
    images, labels = [], []
    for category in categories:
        label = categories.index(category)
        folder = os.path.join(data_dir, category)
        if not os.path.isdir(folder):
            print(f"WARNING: missing folder for category '{category}': {folder}")
            continue
        for fname in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            if img.shape != (IMG_SIZE, IMG_SIZE):
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            images.append(img)
            labels.append(label)
    return np.array(images, dtype=np.uint8), np.array(labels, dtype=np.int64)

In [ ]:
print("Loading dataset...")
X_raw, y = load_raw_data(DATA_DIR, CATEGORIES)
print(f"Loaded {len(X_raw)} images across {len(set(y))} classes.")

#### Stratified split

In [ ]:
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.3, random_state=SEED, stratify=y
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)
print(f"\nTrain: {len(X_train_raw)}  Val: {len(X_val_raw)}  Test: {len(X_test_raw)}")

#### Class Weights

In [ ]:
class_weight_values = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train), y=y_train
)
class_weight_dict = {cls: w for cls, w in zip(np.unique(y_train), class_weight_values)}
print("\nClass weights:", {CATEGORIES[k]: round(v, 2) for k, v in class_weight_dict.items()})

#### Augumentation function

In [ ]:
def augment_image(img):
    out = img.copy()

    # --- Stroke width variation (thicken/thin strokes) ---
    # Dark stroke on light background: erode -> thicker strokes, dilate -> thinner strokes.
    r = random.random()
    if r < 0.35:
        k = random.choice([2, 3])
        kernel = np.ones((k, k), np.uint8)
        out = cv2.erode(out, kernel, iterations=1)   # thicker
    elif r < 0.55:
        k = random.choice([2, 3])
        kernel = np.ones((k, k), np.uint8)
        out = cv2.dilate(out, kernel, iterations=1)   # thinner
    # else: leave stroke width as-is

    # --- Pose variation: small rotation, shift, zoom ---
    h, w = out.shape
    angle = random.uniform(-12, 12)
    scale = random.uniform(0.9, 1.1)
    tx = random.uniform(-2, 2)
    ty = random.uniform(-2, 2)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty
    out = cv2.warpAffine(out, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=255)

    return out

In [ ]:
class AugmentedSequence(Sequence):

    def __init__(self, images, labels, batch_size, augment=True, shuffle=True):
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(images))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.images) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_imgs = self.images[batch_idx]
        batch_labels = self.labels[batch_idx]

        if self.augment:
            batch_imgs = np.array([augment_image(img) for img in batch_imgs])

        batch_imgs = batch_imgs.astype('float32') / 255.0
        batch_imgs = np.expand_dims(batch_imgs, axis=-1)
        return batch_imgs, batch_labels

In [ ]:
def normalize_plain(images):
    images = images.astype('float32') / 255.0
    return np.expand_dims(images, axis=-1)

In [ ]:
train_gen = AugmentedSequence(X_train_raw, y_train, BATCH_SIZE, augment=True, shuffle=True)
X_val = normalize_plain(X_val_raw)
X_test = normalize_plain(X_test_raw)

#### Model

In [ ]:
odel = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(CATEGORIES), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

#### Train

In [ ]:
history = model.fit(
    train_gen,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

#### Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nOverall test accuracy: {test_acc:.4f}  (loss: {test_loss:.4f})")

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nPer-class report:")
print(classification_report(y_test, y_pred, target_names=CATEGORIES, digits=3, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=range(len(CATEGORIES)))
fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CATEGORIES)
disp.plot(ax=ax, cmap='Blues', xticks_rotation='vertical', colorbar=False)
plt.title('Confusion matrix (test set)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
print("\nSaved confusion_matrix.png -- check it for which specific classes are weak.")
plt.show()

In [ ]:
model.save('equation_reader_v2.keras')
print("\nSaved model to equation_reader_v2.keras")
print("Point your pipeline notebook's load_model(...) at this new file to use it.")